# Homework 12 Part 2: Knowledge Distillation on AG News
SID4 = 5330, SEED = 5330, HP_ID = 2 (HP_ID adds no extra setting in this homework).
Teacher: `teacher_model` (BERT base, loaded locally). Student: `distilbert-base-uncased`. Hardware: Apple M4 GPU (MPS) for both models.

In [ ]:
import os, time, gc, torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 5330
torch.manual_seed(SEED); np.random.seed(SEED)
dev = "mps"
TEACHER, STUDENT = "teacher_model", "distilbert-base-uncased"

ds = load_dataset("fancyzhx/ag_news")
names = ds["train"].features["label"].names
tok = AutoTokenizer.from_pretrained(STUDENT)
assert tok.get_vocab() == AutoTokenizer.from_pretrained(TEACHER).get_vocab()

def enc(split):
    e = tok(list(ds[split]["text"]), truncation=True, max_length=128, padding="max_length", return_tensors="pt")
    return e.input_ids, e.attention_mask, torch.tensor(ds[split]["label"])
Xtr, Mtr, Ytr = enc("train"); Xte, Mte, Yte = enc("test")

def forward(model, X, M, b):
    n = M[b].sum(1).max()
    return model(input_ids=X[b, :n].to(dev), attention_mask=M[b, :n].to(dev)).logits

@torch.no_grad()
def logits(model, X, M, bs=256):
    model.eval()
    return torch.cat([forward(model, X, M, slice(i, i + bs)).cpu() for i in range(0, len(X), bs)])

## Teacher logits for all training samples

In [ ]:
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER).to(dev)
t_logits = logits(teacher, Xtr, Mtr)
print("Teacher logits:", tuple(t_logits.shape), "train accuracy:", round((t_logits.argmax(1) == Ytr).float().mean().item(), 4))
del teacher

## Distillation: loss = alpha * CE + (1 - alpha) * KLD, temperature sweep then alpha sweep

In [ ]:
def train(T, alpha, bs=64, lr=5e-5):
    torch.manual_seed(SEED)
    s = AutoModelForSequenceClassification.from_pretrained(STUDENT, num_labels=4).to(dev).train()
    opt, log = torch.optim.AdamW(s.parameters(), lr=lr), []
    perm = torch.randperm(len(Xtr))
    for step, i in enumerate(range(0, len(perm), bs)):
        b = perm[i:i + bs]
        out = forward(s, Xtr, Mtr, b)
        ce = F.cross_entropy(out, Ytr[b].to(dev))
        kld = F.kl_div(F.log_softmax(out / T, -1), F.softmax(t_logits[b].to(dev) / T, -1), reduction="batchmean") * T * T
        loss = alpha * ce + (1 - alpha) * kld
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 50 == 0: log.append((step, loss.item(), ce.item(), kld.item()))
        if step % 300 == 0: print(f"T={T} alpha={alpha} step {step:4d} loss {loss.item():.4f} ce {ce.item():.4f} kld {kld.item():.4f}")
    acc = (logits(s, Xte, Mte).argmax(1) == Yte).float().mean().item()
    path = f"students/T{T}_a{alpha}"; s.save_pretrained(path)
    print(f"T={T} alpha={alpha} test accuracy {acc:.4f}\n")
    return dict(T=T, alpha=alpha, acc=acc, log=log, path=path)

t_runs = [train(T, 0.5) for T in (1, 2, 5, 10)]
best_run = max(t_runs, key=lambda r: r["acc"]); best_T = best_run["T"]
a_runs = [train(best_T, 0.25), best_run, train(best_T, 0.75)]  # alpha=0.5 at best_T is the same seeded run as in the sweep

In [ ]:
for r in t_runs:
    plt.plot(*zip(*[(l[0], l[1]) for l in r["log"]]), label=f"T={r['T']}")
plt.xlabel("step"); plt.ylabel("training loss"); plt.yscale("log"); plt.legend(); plt.title("Loss trend, alpha=0.5"); plt.show()
print("Best temperature:", best_T)
print(pd.DataFrame(t_runs)[["T", "alpha", "acc"]].round(4).to_string(index=False), "\n")
print(pd.DataFrame(a_runs)[["T", "alpha", "acc"]].round(4).to_string(index=False))

## Teacher vs best student on the AG News test set

In [ ]:
def size_mb(path):
    return sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) / 1e6

@torch.no_grad()
def measure(path, n=100):
    gc.collect(); torch.mps.empty_cache()
    m = AutoModelForSequenceClassification.from_pretrained(path).to(dev).eval()
    peak = [0]
    hooks = [mod.register_forward_hook(lambda *a: peak.__setitem__(0, max(peak[0], torch.mps.current_allocated_memory()))) for mod in m.modules()]
    s = time.perf_counter(); pred = logits(m, Xte, Mte).argmax(1); total = time.perf_counter() - s
    for h in hooks: h.remove()
    t = []
    for i in range(n + 5):
        s = time.perf_counter(); forward(m, Xte, Mte, slice(i, i + 1)).cpu(); t.append((time.perf_counter() - s) * 1000)
    p50, p95 = np.percentile(t[5:], [50, 95])
    del m
    return pred, [round((pred == Yte).float().mean().item(), 4), f"{p50:.1f} / {p95:.1f}", round(peak[0] / 1e9, 3), round(size_mb(path), 1), round(total, 1)]

best = max(t_runs + a_runs, key=lambda r: r["acc"])
t_pred, t_row = measure(TEACHER)
s_pred, s_row = measure(best["path"])
print("Best student:", best["path"])
print(pd.DataFrame({"Teacher (BERT base)": t_row, "Best distilled student": s_row},
                   index=["Accuracy", "p50 / p95 latency (ms)", "Peak memory (GB)", "Size on disk (MB)", "Test set inference time (s)"]).T.to_string())

## Where the compression hurts

In [ ]:
pc = pd.DataFrame({"teacher": [(t_pred[Yte == c] == c).float().mean().item() for c in range(4)],
                   "student": [(s_pred[Yte == c] == c).float().mean().item() for c in range(4)]}, index=names)
pc["drop"] = pc.teacher - pc.student
print(pc.round(4).to_string())
c = int(pc["drop"].values.argmax())
print("Most degraded class:", names[c])
for i in ((Yte == c) & (t_pred == c) & (s_pred != c)).nonzero().flatten()[:5].tolist():
    print(f"\n[student said {names[s_pred[i]]}] {ds['test'][i]['text']}")